# Part 4 · Notebook 04 — Live streams, bar building and the cache

**Sessions:** S7 (Live streaming data) · S8 (The unified `DataHandler`) · [Lesson plan](../../docs/lessons/PART_04_BROKER_CONNECTIVITY.md) · graded labs in [`labs/part04/`](../../labs/part04/)

**You will:**
1. Build time bars from a stream of ticks.
2. See why a bar builder needs a timer, not just ticks.
3. Work out which date ranges a read-through cache still has to download.
4. Spot a stale feed before a strategy trades on it.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
Nothing here connects to a broker: the account rows, bars, ticks and order events are synthetic, shaped like what `ib_async` and `alpaca-py` return.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p4lib.py is in notebooks/part04/
    sys.path.insert(0, str(d))
from decimal import Decimal
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p4lib as p

p.use_course_style()

## 1. A tick stream with quiet spells

Real streams are bursty: busy minutes with a tick every few hundred milliseconds, then quiet spells with nothing for tens of seconds.

In [ ]:
ticks = p.tick_stream()
t, px = np.array([x[0] for x in ticks]), np.array([x[1] for x in ticks])
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(t, px, ".", ms=3)
ax.set(xlabel="seconds since the open", ylabel="price", title=f"{len(ticks)} ticks in 5 minutes")
plt.show()
gaps = np.diff(t)
print(f"median gap {np.median(gaps):.2f} s, longest gap {gaps.max():.1f} s")

## 2. Ticks → 5-second bars

A bar covers `[start, start + seconds)`, where `start` is `ts` rounded **down** to a multiple of `seconds`. `on_tick` closes the current bar when a tick arrives past its end, then starts or updates a bar. `on_timer(now)` (already written) closes the current bar once `now` has passed its end, even if no tick arrives.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
class MyBarBuilder(p.BarBuilder):
    def on_tick(self, ts: float, price: float, size: int) -> list[dict]:
        out = []
        if self.cur is not None and ts >= self.cur["start"] + self.seconds:
            out.append(self.cur)
            self.cur = None
        if self.cur is None:
            start = ...                           # ✍️ ts rounded down to a multiple of self.seconds
            self.cur = {"start": start, "open": price, "high": price, "low": price, "close": price, "volume": size}
        else:
            ...                                   # ✍️ update high, low and close, and add size to volume
        return out

mine = p.attempt(lambda: p.run_builder(MyBarBuilder(5), ticks, timer_every=1.0)[0])
ref, _ = p.run_builder(p.BarBuilder(5), ticks, timer_every=1.0)
mine = p.check("bar builder", mine, ref)
pd.DataFrame(mine).head()

## 3. Why the timer matters

Without a timer, a bar is only published when the **next** tick arrives. In a quiet spell your strategy sees the bar late, and the last bar before a long silence may never arrive at all.

In [ ]:
with_timer, d1 = p.run_builder(p.BarBuilder(5), ticks, timer_every=1.0)
ticks_only, d2 = p.run_builder(p.BarBuilder(5), ticks, timer_every=None)
fig, ax = plt.subplots()
ax.hist([d1, d2], bins=np.arange(0, 32, 1), label=["with a 1 s timer", "ticks only"])
ax.set(xlabel="seconds between the bar's end and its publication", ylabel="bars", title="Bar publication delay")
ax.legend(); plt.show()
print(f"with timer: {len(with_timer)} bars, max delay {max(d1):.1f} s")
print(f"ticks only: {len(ticks_only)} bars, max delay {max(d2):.1f} s, and the last bar was never published")

## 4. A read-through cache

`DataHandler.get_bars(symbol, start, end)` should read from the local store and download **only the missing ranges**. `have` is a list of `(start, end)` ranges already stored, possibly overlapping and unsorted. Return the sub-ranges of `[start, end)` that no stored range covers, in order.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def missing_ranges(have, start, end):
    out, cur = [], start                          # cur: everything before cur is covered
    for s, e in sorted(have):
        if e <= cur:
            continue                              # entirely behind us
        if s > cur:
            ...                                   # ✍️ a gap from cur to s (but not past end)
        cur = ...                                 # ✍️ now covered up to here
        if cur >= end:
            break
    if cur < end:
        out.append((cur, end))
    return out

cases = [([(5, 10), (8, 15), (20, 25)], 0, 30), ([], 0, 10), ([(0, 100)], 10, 20), ([(12, 18), (0, 4)], 2, 15)]
mine = [p.attempt(missing_ranges, *c) for c in cases]
mine = p.check("missing_ranges", mine, [p.missing_ranges(*c) for c in cases])
mine

Now a research day: 200 requests for random windows of days in a 2-year range. How many downloads does the cache make?

In [ ]:
downloads = []
cache = p.BarCache(lambda a, b: downloads.append((a, b)))
rng = np.random.default_rng(1)
for _ in range(200):
    a = int(rng.integers(0, 700)); cache.get(a, a + int(rng.integers(5, 60)))
print(f"200 requests → {cache.downloads} downloads, {sum(b - a for a, b in downloads)} days fetched in total")
print("without the cache: 200 downloads, and every one of them paced by the broker")

## 5. A stale feed

A connected socket is not a live feed. Track the age of the last tick; if it exceeds a threshold during market hours, stop trading on that symbol and alert. Here, a 10-second threshold during our quiet spells:

In [ ]:
now = np.arange(0, 300)
last_tick = np.array([t[t <= s].max() if (t <= s).any() else np.nan for s in now])
age = now - last_tick
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.plot(now, age); ax.axhline(10, color=p.PALETTE[7], ls="--", label="stale threshold")
ax.set(xlabel="seconds since the open", ylabel="age of last tick (s)", title="Feed staleness"); ax.legend(); plt.show()
print(f"stale for {(age > 10).sum()} of 300 seconds; the quiet spells are exactly where a stale check matters")

## Wrap-up

* Bars close on time only if something besides ticks (a timer) closes them.
* A read-through cache downloads only what is missing: fast research and fewer pacing waits.
* Watch tick age, not just the connection state.
* Graded version: `labs/part04/week14_data` (`BarBuilder` with timer flush, `missing_ranges`, `BarCache`).